In [4]:
import pandas as pd
from rapidfuzz import fuzz

INPUT_FILE = "All_articles.xlsx"
OUTPUT_FILE = "deduplicated.xlsx"

df = pd.read_excel(INPUT_FILE)

THRESHOLD = 90

df["ClusterID"] = -1
visited = set()

titles = df["Title_norm"].fillna("").tolist()

cluster_id = 0

# ========= CLUSTERING =========
for i in range(len(df)):
    if i in visited:
        continue

    df.loc[i, "ClusterID"] = cluster_id
    visited.add(i)

    for j in range(i + 1, len(df)):
        if j in visited:
            continue

        score = fuzz.ratio(titles[i], titles[j])

        if score >= THRESHOLD:
            df.loc[j, "ClusterID"] = cluster_id
            visited.add(j)

    cluster_id += 1

# ========= KEEP ONE PER CLUSTER =========
df_dedup = df.sort_values(by=["ClusterID", "Database"]).drop_duplicates(
    subset=["ClusterID"],
    keep="first"
)

# ========= STATS =========
before = len(df)
after = len(df_dedup)
removed = before - after

print("\n===== DÉDUPLICATION REPORT =====")
print(f"Avant suppression : {before}")
print(f"Après suppression : {after}")
print(f"Doublons supprimés : {removed}")
print(f"Taux de réduction : {round((removed/before)*100, 2)}%")

# ========= EXPORT =========
df_dedup.to_excel(OUTPUT_FILE, index=False)

print("\nFichier généré :", OUTPUT_FILE)


===== DÉDUPLICATION REPORT =====
Avant suppression : 242
Après suppression : 239
Doublons supprimés : 3
Taux de réduction : 1.24%

Fichier généré : deduplicated_fuzzy.xlsx
